# Bundesliga predictor

In [1]:
import pandas as pd 
import requests
import numpy as np
from collections import defaultdict
from typing import Dict

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
def get_table(season: int) -> pd.DataFrame:

    url = f"https://api.openligadb.de/getbltable/bl1/{season}"
    data = requests.get(url).json()

    data

    rows = []
    for team in data:
        rows.append({
            "team_name" : team["teamName"],
            "points" : team["points"],
            "win" : team["won"],
            "draw" : team["draw"],
            "loss" : team["lost"],
            "goals_for" : team["goals"],
            "goals_against" : team["opponentGoals"],
            "goal_diff" : team["goalDiff"]
            })
    table = pd.DataFrame(rows)
    table["position"] = table.index + 1
    return table

The API provided by OpenLigaDB is incorrect for Bundesliga seasons prior to 2023/24, so I had to switch. I found a better API than the previous one, allowing us to implement more stats in our final table!! This api does not provide a league table, but gives detailed of all matches of each matchday. From there, we can compute the wins, losses, draws along with goal difference. 

In [3]:
def get_BL_matches(season: int) -> pd.DataFrame :
    url = f"https://raw.githubusercontent.com/openfootball/football.json/master/{season}-{season%100 + 1}/de.1.json"
    data = requests.get(url).json()

    rows = []
    for match in data["matches"]:
        rows.append({
            "matchday": match["round"],
            "date": match["date"],
            "home": match["team1"],
            "away": match["team2"],
            "score_home": match["score"]["ft"][0],
            "score_away": match["score"]["ft"][1]
        })
    matches = pd.DataFrame(rows)
    return matches

In [4]:
def create_table(season: int) -> pd.DataFrame:

    season_matches = get_BL_matches(season)

    teams: Dict[str, Dict[str, int]] = defaultdict(lambda: {
            "wins": 0,
            "draws": 0,
            "losses": 0,
            "goals_for": 0,
            "goals_against": 0,
            "home_wins": 0,
            "home_losses": 0
        })

    for _, row in season_matches.iterrows():
        home = row["home"]
        away = row["away"]

        home_goals = row["score_home"]
        away_goals = row["score_away"]

        teams[home]["goals_for"] += home_goals
        teams[home]["goals_against"] += away_goals
        teams[away]["goals_for"] += away_goals
        teams[away]["goals_against"] += home_goals

        if home_goals > away_goals:
            teams[home]["wins"] += 1
            teams[home]["home_wins"] += 1
            teams[away]["losses"] += 1
        elif home_goals < away_goals:
            teams[away]["wins"] += 1
            teams[home]["home_losses"] += 1
            teams[home]["losses"] += 1
        else:
            teams[away]["draws"] += 1
            teams[home]["draws"] += 1

    data = []

    for team, statistics in teams.items():
        goal_diff = statistics["goals_for"] - statistics["goals_against"]
        points = statistics["wins"] * 3 + statistics["draws"] 

        data.append({
            "team_name" : team,
            "points" : points,
            "win" : statistics["wins"],
            "draw" : statistics["draws"],
            "loss" : statistics["losses"],
            "goals_for" : statistics["goals_for"],
            "goals_against" : statistics["goals_against"],
            "goal_diff" : goal_diff,
            "home_wins" : statistics["home_wins"],
            "home_losses" : statistics["home_losses"]
        })

    table = pd.DataFrame(data)

    table = table.sort_values(["points", "goal_diff", "goals_for"], ascending= [False, False, False]).reset_index(drop= True)

    table["position"] = table.index + 1

    return table

In [5]:
def prep_data(start_date : int, end_date : int): #end date should be 2024
    season_tables : Dict[str, pd.DataFrame] = {}
    for year in range(start_date, end_date + 1):
        table = create_table(year)
        season_tables[f"{year}"] = table
    feature_rows = []
    target_rows = []
    for i in range(start_date, end_date):
        prev_table = season_tables[f"{i}"].copy().set_index("team_name")
        curr_table = season_tables[f"{i +1}"].copy().set_index("team_name")
        relegated = prev_table.tail(3)
        promoted_stats = relegated.mean().to_dict()

        for team, row in curr_table.iterrows():
            if team in prev_table.index:
                features = prev_table.loc[team][["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff", "home_wins", "home_losses"]].to_dict()
            else:
                features = {k : promoted_stats[k] for k in ["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff", "home_wins", "home_losses"]}
            feature_rows.append(features)
            target_rows.append(row["position"])
    X_train = pd.DataFrame(feature_rows)
    y_train = pd.Series(target_rows)

    last_feature_rows = []
    last_table = season_tables[f"{end_date}"].copy().set_index("team_name")
    last_relegated = last_table.tail(2)
    last_promoted_stats = last_relegated.mean().to_dict()
    last_teams = last_table.index.tolist()

    last_promoted_teams = ["FC Köln", "Hamburger SV"]
    last_relegated_teams = last_relegated.index.tolist()

    for team in last_teams:
        if team in last_relegated_teams:
            continue
        features = last_table.loc[team][["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff", "home_wins", "home_losses"]].to_dict()
        last_feature_rows.append((team, features))
    for team in last_promoted_teams:
        if team not in last_teams:
            feats = {k : last_promoted_stats[k] for k in ["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff", "home_wins", "home_losses"]}
            last_feature_rows.append((team, feats))
    latest_features_df = pd.DataFrame([feats for _, feats in last_feature_rows],
                                      index=[t for t, _ in last_feature_rows])
    return X_train, y_train, latest_features_df

Now, we start the random forest using Sklearn.

In [6]:
def train_model(X, y) -> Pipeline: 
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("rf", RandomForestClassifier(
            n_estimators=100,
            max_depth=8,
            random_state=42,
            class_weight="balanced"
        ))
    ])
    model.fit(X, y)
    return model

In [7]:
X, y, df = prep_data(2010, 2024)

In [8]:
model = train_model(X, y)

In [9]:
prob = model.predict_proba(df)
classes = model.named_steps["rf"].classes_
exp_positions = prob.dot(classes)
prediction_df = pd.DataFrame({
        "team": df.index,
        "expected_position": exp_positions
        })

In [10]:
prediction_df.sort_values(["expected_position"], ascending=True)
prediction_df["position"] = prediction_df.index +1

In [11]:
prediction_df

,team,expected_position,position
0,FC Bayern München,1.440000,1
1,Bayer 04 Leverkusen,5.685274,2
2,Eintracht Frankfurt,6.424678,3
3,Borussia Dortmund,6.360707,4
4,SC Freiburg,7.427531,5
5,1. FSV Mainz 05,7.943711,6
6,RB Leipzig,9.105074,7
7,SV Werder Bremen,9.431840,8
8,VfB Stuttgart,9.109896,9
9,Borussia Mönchengladbach,10.359112,10


I think I could add some more stats. Some useful aspects could be:
- squad price (total cost of squad at start of season)
- squad rating (avg. rating of starting XI) 

We can get this using a kaggle dataset. 

In [12]:
clubs = pd.read_csv("data/clubs.csv")
player_val = pd.read_csv("data/player_valuations.csv")
players = pd.read_csv("data/players.csv")

In [13]:
players[players["current_club_id"] == 16]

,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,foot,height_in_cm,contract_expiration_date,agent_name,image_url,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,left,190.0,NaN,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0
17,410,Sebastian,Kehl,Sebastian Kehl,2014,16,sebastian-kehl,Germany,Fulda,Germany,...,left,187.0,NaN,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/sebastian-kehl...,L1,Borussia Dortmund,1000000.0,9000000.0
40,1073,Manuel,Friedrich,Manuel Friedrich,2013,16,manuel-friedrich,Germany,Bad Kreuznach,Germany,...,NaN,NaN,NaN,Dr. Michael Becker,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/manuel-friedri...,L1,Borussia Dortmund,500000.0,5500000.0
100,2150,Oliver,Kirch,Oliver Kirch,2014,16,oliver-kirch,Germany,Soest,Germany,...,right,182.0,NaN,Dirk Hebel,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/oliver-kirch/p...,L1,Borussia Dortmund,500000.0,1800000.0
109,2374,Patrick,Owomoyela,Patrick Owomoyela,2012,16,patrick-owomoyela,Germany,Hamburg,Germany,...,right,187.0,NaN,Uwe Kathmann,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/patrick-owomoy...,L1,Borussia Dortmund,200000.0,5000000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32562,1009446,Tyler,Meiser,Tyler Meiser,2024,16,tyler-meiser,NaN,NaN,Germany,...,right,189.0,2028-06-30 00:00:00,SBE Management AG,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/tyler-meiser/p...,L1,Borussia Dortmund,50000.0,50000.0
32833,1045762,Jordi,Paulina,Jordi Paulina,2024,16,jordi-paulina,Netherlands,Odijk,Curacao,...,right,191.0,2030-06-30 00:00:00,NACR-Sports,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/jordi-paulina/...,L1,Borussia Dortmund,250000.0,250000.0
32989,1058359,Samuele,Inácio,Samuele Inácio,2024,16,samuele-inacio,Italy,Bergamo,Italy,...,right,176.0,2027-06-30 00:00:00,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/samuele-inacio...,L1,Borussia Dortmund,NaN,NaN
33208,1074990,Luca,Reggiani,Luca Reggiani,2025,16,luca-reggiani,Italy,Modena,Italy,...,right,194.0,2027-06-30 00:00:00,TMP SOCCER srl,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/luca-reggiani/...,L1,Borussia Dortmund,1000000.0,1000000.0


In [14]:
player_val

,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
0,405973,2000-01-20,150000,Unknown,3057,BE1
1,342216,2001-07-20,100000,Unknown,1241,SC1
2,3132,2003-12-09,400000,Dynamo Kyiv,126,TR1
3,6893,2003-12-15,900000,Galatasaray,984,GB1
4,12359,2004-03-11,250000,RC Lens B,8152,NaN
...,...,...,...,...,...,...
435438,1175607,2026-03-03,100000,Kirklarelispor,27603,NaN
435439,1181950,2026-03-03,10000,Arnavutköy Belediyesi FSK,45493,NaN
435440,1256115,2026-03-03,100000,Karacabey Belediye Spor,20699,NaN
435441,1303292,2026-03-03,25000,Ankara Demirspor,12524,NaN


# Adding additional stats

I want to fetch the club id's for all the clubs in the bundesliga over the seasons I'm considering. To do so, I can check which teams are problematic, and then create a manual mapping.

In [15]:
name_mapping = {
    "SC Freiburg" : "Sport-Club Freiburg",
    "1. FSV Mainz 05" : "1. Fußball- und Sportverein Mainz 05",
    "RB Leipzig" : "RasenBallsport Leipzig",
    "SV Werder Bremen" : "Sportverein Werder Bremen von 1899",
    "VfB Stuttgart" : "Verein für Bewegungsspiele Stuttgart 1893",
    "Borussia Mönchengladbach" : "Borussia Verein für Leibesübungen 1900 Mönchengladbach",
    "VfL Wolfsburg" : "Verein für Leibesübungen Wolfsburg",
    "FC Augsburg" : "Fußball-Club Augsburg 1907",
    "1. FC Union Berlin" : "1. Fußballclub Union Berlin",
    "FC St. Pauli 1910" : "Fußball-Club St. Pauli von 1910",
    "TSG 1899 Hoffenheim" : "Turn- und Sportgemeinschaft 1899 Hoffenheim Fußball-Spielbetriebs",
    "1. FC Heidenheim 1846" : "1. Fußballclub Heidenheim 1846",
    "VfL Bochum 1848" : "VfL Bochum",
    "1. FC Köln" : "1. Fußball-Club Köln",
    "SpVgg Greuther Fürth 1903" : "SpVgg Greuther Fürth",
    "Hamburger SV" : "Hamburger Sport Verein",
    "1. FC Nürnberg" : "1.FC Nuremberg",
    "FC St. Pauli" : "Fußball-Club St. Pauli von 1910",
    "Bayer Leverkusen" : "Bayer 04 Leverkusen Fußball",
    "Bor. Mönchengladbach" : "Borussia Verein für Leibesübungen 1900 Mönchengladbach",
    "1. FC Kaiserslautern" : "test",
    "Bayern München" : "FC Bayern München"
}

teams: Dict[str, int] = {}

for year in range(2010, 2024):
    table = create_table(year)
    for team in table["team_name"]:
        lookup_name = name_mapping.get(team, team)
        
        result = clubs.loc[clubs["name"].str.contains(lookup_name, case=False), "club_id"]
        
        if result.empty:
            print(f"Could not find club_id for: {team}")
            teams[f"{team}"]= np.nan
        else:
            teams[f"{team}"]= result.values[0]

print(teams)

Could not find club_id for: 1. FC Kaiserslautern
Could not find club_id for: 1. FC Kaiserslautern
{'Borussia Dortmund': np.int64(16), 'Bayer Leverkusen': np.int64(15), 'Bayern München': np.int64(27), 'Hannover 96': np.int64(42), '1. FSV Mainz 05': np.int64(39), '1. FC Nürnberg': np.int64(4), '1. FC Kaiserslautern': nan, 'Hamburger SV': np.int64(41), 'SC Freiburg': np.int64(60), '1. FC Köln': np.int64(3), '1899 Hoffenheim': np.int64(533), 'VfB Stuttgart': np.int64(79), 'Werder Bremen': np.int64(86), 'FC Schalke 04': np.int64(33), 'VfL Wolfsburg': np.int64(82), 'Bor. Mönchengladbach': np.int64(18), 'Eintracht Frankfurt': np.int64(24), 'FC St. Pauli': np.int64(35), 'FC Augsburg': np.int64(167), 'Hertha BSC': np.int64(44), 'Fortuna Düsseldorf': np.int64(38), 'SpVgg Greuther Fürth': np.int64(65), 'Eintracht Braunschweig': np.int64(23), 'SC Paderborn 07': np.int64(127), 'FC Ingolstadt 04': np.int64(4795), 'SV Darmstadt 98': np.int64(105), 'RB Leipzig': np.int64(23826), '1. FC Union Berlin': 

In [16]:
clubs

,club_id,club_code,name,domestic_competition_id,total_market_value,squad_size,average_age,foreigners_number,foreigners_percentage,national_team_players,stadium_name,stadium_seats,net_transfer_record,coach_name,last_season,filename,url
0,10,arminia-bielefeld,Arminia Bielefeld,L1,NaN,27,25.3,15,55.6,4,SchücoArena,26515,+€5.90m,NaN,2021,../data/raw/transfermarkt-scraper/2021/clubs.j...,https://www.transfermarkt.co.uk/arminia-bielef...
1,10004,paris-fc,Paris Football Club,FR1,NaN,31,28.5,17,54.8,9,Stade Jean Bouin,19904,€-72.30m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/paris-fc/start...
2,1003,leicester-city,Leicester City,GB1,NaN,29,25.8,17,58.6,10,King Power Stadium,32259,+€57.30m,NaN,2024,../data/raw/transfermarkt-scraper/2024/clubs.j...,https://www.transfermarkt.co.uk/leicester-city...
3,1005,us-lecce,Unione Sportiva Lecce,IT1,NaN,27,25.0,23,85.2,10,Ettore Giardiniero,31559,+€8.62m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/us-lecce/start...
4,1010,fc-watford,Watford FC,GB1,NaN,30,26.3,24,80.0,12,Vicarage Road,21577,+€42.02m,NaN,2021,../data/raw/transfermarkt-scraper/2021/clubs.j...,https://www.transfermarkt.co.uk/fc-watford/sta...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
446,985,manchester-united,Manchester United Football Club,GB1,NaN,26,25.9,19,73.1,14,Old Trafford,74879,€-176.50m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/manchester-uni...
447,987,motherwell-fc,Motherwell Football Club,SC1,NaN,24,26.9,13,54.2,3,Fir Park,13677,+€5.22m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/motherwell-fc/...
448,989,afc-bournemouth,Association Football Club Bournemouth,GB1,NaN,26,25.4,20,76.9,12,Vitality Stadium,11307,+€130.31m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/afc-bournemout...
449,993,fc-cordoba,Córdoba CF,ES1,NaN,24,27.9,4,16.7,0,Nuevo Arcángel,21822,+-0,NaN,2014,../data/raw/transfermarkt-scraper/2014/clubs.j...,https://www.transfermarkt.co.uk/fc-cordoba/sta...


In [17]:
L1 = clubs[clubs["domestic_competition_id"] == "L1"]

L1

,club_id,club_code,name,domestic_competition_id,total_market_value,squad_size,average_age,foreigners_number,foreigners_percentage,national_team_players,stadium_name,stadium_seats,net_transfer_record,coach_name,last_season,filename,url
0,10,arminia-bielefeld,Arminia Bielefeld,L1,NaN,27,25.3,15,55.6,4,SchücoArena,26515,+€5.90m,NaN,2021,../data/raw/transfermarkt-scraper/2021/clubs.j...,https://www.transfermarkt.co.uk/arminia-bielef...
14,105,sv-darmstadt-98,SV Darmstadt 98,L1,NaN,27,25.6,13,48.1,1,Merck-Stadion am Böllenfalltor,17810,+€3.05m,NaN,2023,../data/raw/transfermarkt-scraper/2023/clubs.j...,https://www.transfermarkt.co.uk/sv-darmstadt-9...
65,127,sc-paderborn-07,SC Paderborn 07,L1,NaN,26,25.7,5,19.2,1,Home Deluxe Arena,15000,€-450k,NaN,2019,../data/raw/transfermarkt-scraper/2019/clubs.j...,https://www.transfermarkt.co.uk/sc-paderborn-0...
95,15,bayer-04-leverkusen,Bayer 04 Leverkusen Fußball,L1,NaN,26,26.7,21,80.8,14,BayArena,30210,+€32.50m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/bayer-04-lever...
103,16,borussia-dortmund,Borussia Dortmund,L1,NaN,25,26.5,13,52.0,12,SIGNAL IDUNA PARK,81365,€-23.20m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/borussia-dortm...
109,167,fc-augsburg,Fußball-Club Augsburg 1907,L1,NaN,28,25.5,19,67.9,9,WWK ARENA,30660,€-8.25m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/fc-augsburg/st...
115,18,borussia-monchengladbach,Borussia Verein für Leibesübungen 1900 Mönchen...,L1,NaN,28,25.8,15,53.6,13,Stadion im Borussia-Park,54042,+€10.65m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/borussia-monch...
130,2036,1-fc-heidenheim-1846,1. Fußballclub Heidenheim 1846,L1,NaN,29,26.2,7,24.1,3,Voith-Arena,15000,+€7.50m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/1-fc-heidenhei...
143,23,eintracht-braunschweig,Eintracht Braunschweig,L1,NaN,30,25.6,11,36.7,2,EINTRACHT-Stadion,23325,+€750k,NaN,2013,../data/raw/transfermarkt-scraper/2013/clubs.j...,https://www.transfermarkt.co.uk/eintracht-brau...
152,23826,rasenballsport-leipzig,RasenBallsport Leipzig,L1,NaN,31,23.4,21,67.7,13,Red Bull Arena,47069,+€38.05m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/rasenballsport...
